In [1]:
import pandas as pd

In [ ]:
pollutants= pd.read_csv('fhz_pollutants_1h.csv')

In [3]:
# Re-examine the original end_time range before datetime conversion
print(pd.to_datetime(pollutants['end_time'], format='mixed').sort_values().tail(20))

420486   NaT
420487   NaT
420488   NaT
420489   NaT
420490   NaT
420491   NaT
420492   NaT
420493   NaT
420494   NaT
420495   NaT
420496   NaT
420497   NaT
420498   NaT
420499   NaT
420500   NaT
420501   NaT
420502   NaT
420503   NaT
420504   NaT
420505   NaT
Name: end_time, dtype: datetime64[ns]


In [4]:
temp= pd.read_csv('fhz_temperature_1h.csv')

In [5]:
wind= pd.read_csv('fhz_windspeed.csv')

In [6]:
print(pollutants.columns.tolist())
print(wind.columns.tolist())
print(temp.columns.tolist())

['end_time', 'station', 'pm10_1h', 'pm25_1h', 'so2_1h', 'no2_1h', 'o3_1h', 'co_1h', 'city', 'year']
['year', 'month', 'day', 'hour', 'wind_speed', 'city']
['year', 'month', 'day', 'hour', 'temperature', 'city']


In [7]:
print(wind['hour'].unique()[:10])
print(wind['hour'].dtype)

print(temp['hour'].unique()[:10])
print(temp['hour'].dtype)

['00:00' '00:10' '00:20' '00:30' '00:40' '00:50' '01:00' '01:10' '01:20'
 '01:30']
object
['00:00' '01:00' '02:00' '03:00' '04:00' '05:00' '06:00' '07:00' '08:00'
 '09:00']
object


In [8]:
wind.groupby(['year', 'month', 'day', 'hour', 'city']).size().describe()

,0
count,1.173080e+06
mean,1.998362e+00
std,4.044427e-02
min,1.000000e+00
25%,2.000000e+00
50%,2.000000e+00
75%,2.000000e+00
max,2.000000e+00


In [9]:
wind['hour'] = wind['hour'].str.split(':').str[0].astype(int)

wind_hourly = (
    wind.groupby(['year', 'month', 'day', 'hour', 'city'], as_index=False)
        .agg(wind_speed=('wind_speed', 'mean'))
)

temp['hour'] = temp['hour'].str.split(':').str[0].astype(int)

In [ ]:
# Build a proper datetime column in `pollutants` from end_time
pollutants['datetime'] = pd.to_datetime(pollutants['end_time'], format='mixed')
pollutants['month'] = pollutants['datetime'].dt.month
pollutants['day']   = pollutants['datetime'].dt.day
pollutants['hour']  = pollutants['datetime'].dt.hour
pollutants['year']  = pollutants['datetime'].dt.year  # overwrite to be safe/consistent

# Build matching datetime columns in wind & temp
for df in (wind, temp):
    df['datetime'] = pd.to_datetime(
        dict(year=df['year'], month=df['month'], day=df['day'], hour=df['hour'])
    )

# Merge wind + temp first 
weather = wind.merge(
    temp,
    on=['year', 'month', 'day', 'hour', 'city', 'datetime'],
    how='inner'
)

# Merge weather into pollutants (many stations share one city/hour)
df = pollutants.merge(
    weather,
    on=['year', 'month', 'day', 'hour', 'city'],
    how='left',
    suffixes=('', '_weather')
)

if 'datetime_weather' in df.columns:
    df = df.drop(columns=['datetime_weather'])


# Add season column
def month_to_season(month):
    if month in (12, 1, 2):
        return 'winter'
    elif month in (3, 4, 5):
        return 'spring'
    elif month in (6, 7, 8):
        return 'summer'
    else:
        return 'autumn'

df['season'] = df['month'].apply(month_to_season)
print(f"pollutants rows: {len(pollutants)}")
print(f"merged rows: {len(df)}")
df.head()

pollutants rows: 2304676
merged rows: 4582444


,end_time,station,pm10_1h,pm25_1h,so2_1h,no2_1h,o3_1h,co_1h,city,year,datetime,month,day,hour,wind_speed,temperature,season
0,2021-01-01 00:59:59.998,Ambasada,NaN,105.0,NaN,NaN,NaN,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,122.0,0.4,winter
1,2021-01-01 00:59:59.998,Ambasada,NaN,105.0,NaN,NaN,NaN,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,1.3,0.4,winter
2,2021-01-01 00:59:59.998,Bihac,NaN,NaN,NaN,NaN,NaN,NaN,Bihac,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,325.0,0.4,winter
3,2021-01-01 00:59:59.998,Bihac,NaN,NaN,NaN,NaN,NaN,NaN,Bihac,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,1.6,0.4,winter
4,2021-01-01 00:59:59.998,Bjelave,119.074997,NaN,NaN,13.9008,21.845301,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,122.0,0.4,winter


In [11]:
temp.groupby(['year', 'month', 'day', 'hour', 'city']).size().describe()

,0
count,316183.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [ ]:
for df in (wind_hourly, temp):
    df['datetime'] = pd.to_datetime(
        dict(year=df['year'], month=df['month'], day=df['day'], hour=df['hour'])
    )

weather = wind_hourly.merge(
    temp,
    on=['year', 'month', 'day', 'hour', 'city', 'datetime'],
    how='inner'
)
print(f"weather rows: {len(weather)}")
print(f"weather unique keys: {weather.drop_duplicates(subset=['year','month','day','hour','city']).shape[0]}")

pollutants['datetime'] = pd.to_datetime(pollutants['end_time'], format='ISO8601')
pollutants['month'] = pollutants['datetime'].dt.month
pollutants['day']   = pollutants['datetime'].dt.day
pollutants['hour']  = pollutants['datetime'].dt.hour
pollutants['year']  = pollutants['datetime'].dt.year

# final merge
df = pollutants.merge(
    weather,
    on=['year', 'month', 'day', 'hour', 'city'],
    how='left',
    suffixes=('', '_weather')
)
if 'datetime_weather' in df.columns:
    df = df.drop(columns=['datetime_weather'])

print(f"pollutants: {len(pollutants)}")
print(f"final merged: {len(df)}")

weather rows: 261826
weather unique keys: 261826
pollutants: 2304676
final merged: 2304676


In [ ]:
missing_weather = df['wind_speed'].isna().sum()
print(f"rows missing wind_speed: {missing_weather} ({missing_weather / len(df):.1%})")

missing_temp = df['temperature'].isna().sum()
print(f"rows missing temperature: {missing_temp} ({missing_temp / len(df):.1%})")

df[df['wind_speed'].isna()].groupby('city').size().sort_values(ascending=False)

rows missing wind_speed: 1768289 (76.7%)
rows missing temperature: 1768289 (76.7%)


,0
city,
Zenica,289324
Kakanj,140401
Jajce,78937
Sarajevo,71483
Zivinice,61368
Tuzla,59894
Maglaj,52609
Lukavac,52609
Visoko,43849


In [15]:
pollutant_cities = set(pollutants['city'].unique())
weather_cities = set(weather['city'].unique())

print("In pollutants but not weather:", pollutant_cities - weather_cities)
print("In weather but not pollutants:", weather_cities - pollutant_cities)

In pollutants but not weather: {'Gorazde', 'Zenica', 'Vares', nan, 'Jajce', 'Zivinice', 'Visoko', 'Lukavac', 'Maglaj', 'Tesanj', 'Travnik', 'Kakanj'}
In weather but not pollutants: {'Gradacac', 'Bugojno'}


In [16]:
print(sorted(weather['city'].unique()))

['Bihac', 'Bugojno', 'Gradacac', 'Livno', 'Mostar', 'Sarajevo', 'Tuzla']


In [17]:
import math

for c in sorted(pollutant_cities, key=lambda x: (isinstance(x, float), x)):
    print(repr(c))
print("---")
for c in sorted(weather_cities, key=lambda x: (isinstance(x, float), x)):
    print(repr(c))

'Bihac'
'Gorazde'
'Jajce'
'Kakanj'
'Livno'
'Lukavac'
'Maglaj'
'Mostar'
'Sarajevo'
'Tesanj'
'Travnik'
'Tuzla'
'Vares'
'Visoko'
'Zenica'
'Zivinice'
nan
---
'Bihac'
'Bugojno'
'Gradacac'
'Livno'
'Mostar'
'Sarajevo'
'Tuzla'


In [ ]:
pollutants['city'] = pollutants['city'].str.strip()
pollutants['datetime'] = pd.to_datetime(pollutants['end_time'], format='mixed', errors='coerce')
pollutants['year']  = pollutants['datetime'].dt.year
pollutants['month'] = pollutants['datetime'].dt.month
pollutants['day']   = pollutants['datetime'].dt.day
pollutants['hour']  = pollutants['datetime'].dt.hour

# Drop 2025 from pollutants
pollutants = pollutants[~(pollutants['year'] == 2025.0)].copy()

wind['city'] = wind['city'].str.strip()
if wind['hour'].dtype == object:
    wind['hour'] = wind['hour'].str.split(':').str[0].astype(int)

wind_hourly = (
    wind.groupby(['year', 'month', 'day', 'hour', 'city'], as_index=False)
        .agg(wind_speed=('wind_speed', 'mean'))
)

# ------------------------------------------------------------------
temp['city'] = temp['city'].str.strip()
if temp['hour'].dtype == object:
    temp['hour'] = temp['hour'].str.split(':').str[0].astype(int)

temp = temp.drop(columns=['datetime'], errors='ignore')

# Merge wind + temp -> weather
weather = wind_hourly.merge(
    temp,
    on=['year', 'month', 'day', 'hour', 'city'],
    how='inner'
)
assert len(weather) == weather.drop_duplicates(subset=['year','month','day','hour','city']).shape[0], \
    "weather has duplicate keys!"

# Drop 2020 from weather
weather = weather[~(weather['year'] == 2020.0)].copy()

# 5) Restrict pollutants to weather-covered cities 
covered_cities = {'Bihac', 'Livno', 'Mostar', 'Sarajevo', 'Tuzla'}
pollutants_covered = pollutants[pollutants['city'].isin(covered_cities)].copy()

print(f"pollutants (all cities): {len(pollutants)}")
print(f"pollutants (weather-covered only): {len(pollutants_covered)}")


df = pollutants_covered.merge(
    weather,
    on=['year', 'month', 'day', 'hour', 'city'],
    how='left'
)

print(f"final merged: {len(df)}")
assert len(df) == len(pollutants_covered), "row count changed unexpectedly — check for duplicate keys"


def month_to_season(month):
    if month in (12, 1, 2):
        return 'winter'
    elif month in (3, 4, 5):
        return 'spring'
    elif month in (6, 7, 8):
        return 'summer'
    else:
        return 'autumn'

df['season'] = df['month'].apply(month_to_season)


print("\nMissing values per column:")
print(df.isna().sum())

print("\nRows per city:")
print(df['city'].value_counts())

print("\nDate range:", df['datetime'].min(), "to", df['datetime'].max())

df.head()

pollutants (all cities): 1384941
pollutants (weather-covered only): 587275
final merged: 587275

Missing values per column:
end_time           12
station             0
pm10_1h        286058
pm25_1h        343909
so2_1h         206579
no2_1h         189725
o3_1h          348018
co_1h          355680
city                0
year               12
datetime           12
month              12
day                12
hour               12
wind_speed      50888
temperature     50888
season              0
dtype: int64

Rows per city:
city
Sarajevo    333071
Tuzla       157773
Bihac        35064
Livno        35064
Mostar       26303
Name: count, dtype: int64

Date range: 2021-01-01 00:59:59.998000 to 2024-12-31 23:00:00


,end_time,station,pm10_1h,pm25_1h,so2_1h,no2_1h,o3_1h,co_1h,city,year,datetime,month,day,hour,wind_speed,temperature,season
0,2021-01-01 00:59:59.998,Ambasada,NaN,105.0,NaN,NaN,NaN,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,61.65,0.4,winter
1,2021-01-01 00:59:59.998,Bihac,NaN,NaN,NaN,NaN,NaN,NaN,Bihac,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,163.30,0.4,winter
2,2021-01-01 00:59:59.998,Bjelave,119.074997,NaN,NaN,13.9008,21.845301,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,61.65,0.4,winter
3,2021-01-01 00:59:59.998,Hadzici,NaN,NaN,NaN,NaN,NaN,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,61.65,0.4,winter
4,2021-01-01 00:59:59.998,Hadzici,NaN,NaN,NaN,NaN,NaN,NaN,Sarajevo,2021.0,2021-01-01 00:59:59.998,1.0,1.0,0.0,61.65,0.4,winter


In [34]:
df = df.drop(columns=['end_time'])

In [35]:
# compare year ranges between pollutants (covered cities) and weather
print("pollutants years:", sorted(pollutants_covered['year'].dropna().unique()))
print("weather years:", sorted(weather['year'].unique()))

# is missing weather concentrated in specific years?
df[df['wind_speed'].isna()].groupby('year').size()

# or specific cities?
df[df['wind_speed'].isna()].groupby('city').size()

# or specific hours (sensor downtime pattern)?
df[df['wind_speed'].isna()].groupby('hour').size().sort_values(ascending=False).head(10)

pollutants years: [np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0)]
weather years: [np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0)]


,0
hour,
3.0,4525
21.0,4503
0.0,4473
18.0,4365
15.0,4280
12.0,3816
6.0,3664
9.0,3661
5.0,2317


In [36]:
df.describe()

,no2_1h,o3_1h,co_1h,year,datetime,month,day,hour,wind_speed,temperature
count,397550.000000,239257.000000,231595.000000,587263.000000,587263,587263.000000,587263.000000,587263.000000,536387.000000,536387.000000
mean,20.122517,58.354218,0.630715,2022.389066,2022-11-21 14:32:10.684721664,6.523413,15.729115,11.246545,88.667149,12.107506
min,-4.923020,-3.667600,-0.061786,2021.000000,2021-01-01 00:59:59.998000,1.000000,1.000000,0.000000,0.000000,-14.700000
25%,6.700000,21.837800,0.200260,2021.000000,2021-11-02 21:00:00,4.000000,8.000000,6.000000,53.950000,4.900000
50%,15.150000,50.643002,0.366000,2022.000000,2022-11-05 09:59:59.998000128,7.000000,16.000000,12.000000,74.520000,11.800000
75%,28.426050,86.181999,0.819000,2023.000000,2023-11-22 18:00:00,10.000000,23.000000,18.000000,133.050000,18.500000
max,272.500000,720.700012,10.595600,2024.000000,2024-12-31 23:00:00,12.000000,31.000000,23.000000,183.033333,41.600000
std,18.246984,44.396598,0.708670,1.132682,NaN,3.448539,8.799469,6.935694,46.078718,9.218584


In [37]:
df_pm10 = df.dropna(subset=['pm10_1h', 'wind_speed', 'temperature'])
df_pm25 = df.dropna(subset=['pm25_1h', 'wind_speed', 'temperature'])

print(f"pm10 training rows: {len(df_pm10)}")
print(f"pm25 training rows: {len(df_pm25)}")

pm10 training rows: 286678
pm25 training rows: 211427


In [38]:
df.groupby('station')['pm10_1h'].apply(lambda x: x.isna().mean()).sort_values(ascending=False)
df.groupby('station')['pm25_1h'].apply(lambda x: x.isna().mean()).sort_values(ascending=False)

,pm25_1h
station,
Isedlo,1.000000
Ilijas,1.000000
Tuzla-Trnovac,1.000000
Vijecnica,1.000000
Hadzici,0.999870
Tuzla-Bukinje,0.858978
Otoka,0.752880
Mostar,0.472075
Tuzla-Skver,0.311815


In [39]:
df.to_csv("fhz_data.csv", index=False)